[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C21_Frontier_Pretraining_Course/03_mup_scaling/03_mup_scaling.ipynb)

# 03 · μP 与超参迁移（亲手测出尺度漂移）

目标：用纯 numpy 在玩具 MLP 上**测出** SP 下超参随宽度漂移、μP 下不漂移，做一次 **coordinate check**，验证 attention 用 1/d 的理由。

路线：方差传播基线 → SP 下 Δoutput 随宽度增长(核心) → μP 缩放让 Δoutput O(1) → attention 1/d vs 1/√d → coordinate check → LR transfer 验证 → ✏️ 练习(μP 缩放因子 / coord check / lr transfer / Δy 标度) → 📖 答案 → 🧪 真实 scaling 算账胶囊。

> 心智模型：**μP 让「一步更新对网络输出的冲击力」在所有宽度上恒定 → 最优 lr 恒定 → 可迁移**。我们用一步 SGD 的 Δoutput 当探针。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
print('环境就绪，开始测尺度')

## 1 · 方差传播：为什么初始化方差取 1/fan_in

若 `h = W x`，W 元素方差 σ²、x 各坐标方差 1、fan_in=d_in，则 h 每坐标方差 ≈ `d_in·σ²`。取 `σ²=1/d_in` 让 h 方差≈1(尺度不随 fan_in 漂)。这是所有参数化的共同起点。

In [ ]:
def hidden_var(d_in, d_out, sigma2, n_trials=200):
    '''测 h=Wx 的输出坐标方差。'''
    g = np.random.default_rng(1)
    vars_ = []
    for _ in range(n_trials):
        W = g.standard_normal((d_out, d_in)) * np.sqrt(sigma2)
        x = g.standard_normal(d_in)
        vars_.append(np.var(W @ x))
    return np.mean(vars_)

print('用 sigma2 = 1/d_in (正确缩放):')
for d_in in [16, 64, 256, 1024]:
    v = hidden_var(d_in, 32, 1.0 / d_in)
    print(f'  d_in={d_in:5d}: 输出方差 {v:.3f} (≈1，与 d_in 无关 ✓)')

print('\n用固定 sigma2 = 1 (错误，不随 fan_in 缩放):')
for d_in in [16, 64, 256]:
    v = hidden_var(d_in, 32, 1.0)
    print(f'  d_in={d_in:5d}: 输出方差 {v:.1f} (随 d_in 爆炸 ✗)')

assert abs(hidden_var(1024, 32, 1.0/1024) - 1.0) < 0.2, '1/fan_in 缩放下方差应≈1'
assert hidden_var(256, 32, 1.0) > 100, '固定方差下输出随 fan_in 爆炸'
print('\n✅ 1/fan_in 初始化让激活初始尺度与宽度无关 —— 但这只管「初始」，训练动态是 μP 的战场')

## 2 · SP 下 Δoutput 随宽度增长（本模块核心）

**关键探针**：一步 SGD 后，网络输出的改变量 Δy。两层 MLP `h=relu(W1 x); y=W2 h`，宽度=W1 的行数。

**SP + 固定 lr**：Δy 随宽度增长(网络越宽，同一 lr 冲击越大) → 最优 lr 随宽度往下漂。这就是「不能把小模型 lr 用到大模型」的病根。

In [ ]:
def delta_output(width, param, n_seeds=40, lr=0.05, batch=16, d_in=8):
    '''测一步 SGD 后网络输出的 RMS 改变量 Δy(平均多个 seed)。'''
    deltas = []
    for seed in range(n_seeds):
        g = np.random.default_rng(seed)
        if param == 'sp':
            W1 = g.standard_normal((width, d_in)) * np.sqrt(1/d_in)
            W2 = g.standard_normal((1, width)) * np.sqrt(1/width)   # SP: 1/sqrt(width)
            lr1 = lr; lr2 = lr
        else:  # muP (SGD 版)
            W1 = g.standard_normal((width, d_in)) * np.sqrt(1/d_in)
            W2 = g.standard_normal((1, width)) * (1.0/width)        # muP: readout 初始 ~1/width
            lr1 = lr * width                                        # muP-SGD: 隐藏层 lr 随宽度放大
            lr2 = lr / width                                        # muP-SGD: 输出层 lr 随宽度缩小
        X = g.standard_normal((batch, d_in)); T = g.standard_normal(batch)
        pre = X @ W1.T; H = np.maximum(pre, 0); Y = (H @ W2.T)[:, 0]
        dY = (Y - T) / batch
        gW2 = dY[None, :] @ H
        gH = dY[:, None] * W2
        gpre = gH * (pre > 0)
        gW1 = gpre.T @ X
        W1n = W1 - lr1 * gW1; W2n = W2 - lr2 * gW2
        Yn = (np.maximum(X @ W1n.T, 0) @ W2n.T)[:, 0]
        deltas.append(np.sqrt(np.mean((Yn - Y) ** 2)))
    return np.mean(deltas)

print('SP：Δoutput 随宽度增长(坏) | μP：Δoutput 保持 O(1)(好)\n')
print(f"{'width':>6} {'SP Δy':>10} {'μP Δy':>10}")
sp_vals, mup_vals = [], []
for w in [16, 64, 256, 1024, 4096]:
    s, m = delta_output(w, 'sp'), delta_output(w, 'mup')
    sp_vals.append(s); mup_vals.append(m)
    print(f'{w:6d} {s:10.4f} {m:10.4f}')

# SP 单调增长(从 16 到 4096 涨很多倍)；μP 基本持平
assert sp_vals[-1] > 10 * sp_vals[0], 'SP 的 Δy 应随宽度大幅增长'
assert max(mup_vals) / min(mup_vals) < 3.0, 'μP 的 Δy 应在各宽度基本持平(O(1))'
print('\n✅ 实测：SP 下 Δoutput∝宽度(故最优 lr 随宽度漂)；μP 下 Δoutput=O(1)(故 lr 可迁移)')

> 这就是 μP 的全部秘密的实测证据：**SP 让宽网络对同一 lr 更敏感**(Δy 大)，所以你必须把 lr 调小——最优 lr 随宽度漂移。**μP 通过缩放 init/lr 让 Δy 在所有宽度恒定**，最优 lr 因此不漂移、可迁移。

## 3 · μP 缩放因子：把 SP 转成 μP

把上面的缩放规则封装成函数：给定基准宽度与目标宽度，算出各层 init 方差与 lr 的缩放倍数。这就是 `mup` 库帮你做的事。

In [ ]:
def mup_scaling(base_width, target_width, optimizer='sgd'):
    '''返回从 base_width 缩放到 target_width 时各部分的乘法因子(μP, SGD 版)。'''
    ratio = target_width / base_width
    return {
        'hidden_init_var_mult':  1.0 / ratio,    # 隐藏层 init 方差 ∝ 1/width
        'readout_init_var_mult': 1.0 / ratio**2, # 输出层 init 方差 ∝ 1/width^2
        'hidden_lr_mult':        ratio if optimizer == 'sgd' else 1.0,  # SGD:∝width, Adam:常数
        'readout_lr_mult':       1.0 / ratio,    # 输出层 lr ∝ 1/width
        'attn_logit_scale':      'use 1/d (not 1/sqrt(d))',
    }

print('从宽度 256 缩放到 1024 (4x) 的 μP 因子:')
f = mup_scaling(256, 1024, 'sgd')
for k, v in f.items():
    print(f'  {k:24s}: {v}')

assert mup_scaling(256, 1024)['hidden_init_var_mult'] == 0.25, '4x 宽 -> init 方差 1/4'
assert mup_scaling(256, 1024, 'sgd')['hidden_lr_mult'] == 4.0, 'SGD 下隐藏层 lr ∝ width'
assert mup_scaling(256, 1024, 'adam')['hidden_lr_mult'] == 1.0, 'Adam 下隐藏层 lr 常数'
assert mup_scaling(256, 1024)['readout_lr_mult'] == 0.25, '输出层 lr ∝ 1/width'
print('\n✅ μP 缩放因子：隐藏 init ∝1/width、输出 init ∝1/width²、lr 按优化器调、注意力用 1/d')

## 4 · attention：为什么 μP 用 1/d 而非 1/√d

初始时 q,k **独立** → `q·k ~ √d`，除 √d 拉回 O(1)。但**训练后 q,k 相关** → `q·k ~ d`，只有除 d 才能拉回 O(1)。测给你看。

In [ ]:
def qk_scale(d, correlated, n=300):
    '''测 q·k 的典型绝对值。correlated=True 模拟训练后(q,k 相关)。'''
    g = np.random.default_rng(2)
    vals = []
    for _ in range(n):
        if correlated:
            base = g.standard_normal(d)
            q = base + 0.3 * g.standard_normal(d)   # q,k 共享方向(相关)
            k = base + 0.3 * g.standard_normal(d)
        else:
            q = g.standard_normal(d); k = g.standard_normal(d)  # 独立
        vals.append(abs(q @ k))
    return np.mean(vals)

print('训练后(q,k 相关)的 q·k 尺度:')
print(f"{'d':>6} {'raw q·k':>10} {'/√d':>8} {'/d':>8}")
for d in [16, 64, 256, 1024]:
    raw = qk_scale(d, correlated=True)
    print(f'{d:6d} {raw:10.1f} {raw/np.sqrt(d):8.2f} {raw/d:8.3f}')

# 相关情形下 raw q·k ∝ d；除以 d 保持 O(1)，除以 √d 仍随 √d 漂
raw_small = qk_scale(64, True); raw_big = qk_scale(1024, True)
assert raw_big / raw_small > 8, '相关 q·k 应随 d 线性增长(16x d -> ~16x)'
assert abs(raw_big/1024 - raw_small/64) < 0.2, '除以 d 后在各 d 保持 O(1)'
assert (raw_big/np.sqrt(1024)) / (raw_small/np.sqrt(64)) > 2, '除以 √d 仍随 √d 漂移'
print('\n✅ 训练后 q·k∝d，故 μP 用 1/d 让注意力 logit 在各宽度保持 O(1)(防 softmax 饱和)')

## 5 · coordinate check：验证 μP 实现对不对

标准诊断：在几个宽度上测每层激活/更新的坐标量级，画随宽度的曲线。**μP 正确 → 曲线水平(与宽度无关)；SP → 倾斜(漂移)**。

我们对第 2 节的 Δactivation 做 coordinate check。

In [ ]:
def coord_check_delta_h(width, param, n_seeds=30, lr=0.05, batch=16, d_in=8):
    '''测一步更新后隐藏激活 h 的改变量 Δh 的坐标 RMS(特征学习强度)。'''
    rmss = []
    for seed in range(n_seeds):
        g = np.random.default_rng(seed)
        if param == 'sp':
            W1 = g.standard_normal((width, d_in)) * np.sqrt(1/d_in)
            W2 = g.standard_normal((1, width)) * np.sqrt(1/width)
            lr1 = lr; lr2 = lr
        else:
            W1 = g.standard_normal((width, d_in)) * np.sqrt(1/d_in)
            W2 = g.standard_normal((1, width)) * (1.0/width)
            lr1 = lr * width; lr2 = lr / width
        X = g.standard_normal((batch, d_in)); T = g.standard_normal(batch)
        pre = X @ W1.T; H = np.maximum(pre, 0); Y = (H @ W2.T)[:, 0]
        dY = (Y - T) / batch
        gH = dY[:, None] * W2; gpre = gH * (pre > 0); gW1 = gpre.T @ X
        W1n = W1 - lr1 * gW1
        pre_new = X @ W1n.T; H_new = np.maximum(pre_new, 0)
        rmss.append(np.sqrt(np.mean((H_new - H) ** 2)))   # Δh 坐标 RMS
    return np.mean(rmss)

widths = [16, 64, 256, 1024]
print(f"{'width':>6} {'SP |Δh|':>12} {'μP |Δh|':>12}")
sp_dh, mup_dh = [], []
for w in widths:
    s, m = coord_check_delta_h(w, 'sp'), coord_check_delta_h(w, 'mup')
    sp_dh.append(s); mup_dh.append(m)
    print(f'{w:6d} {s:12.5f} {m:12.5f}')

# coordinate check 判据：μP 的 |Δh| 随宽度近似水平；SP 倾斜
mup_spread = max(mup_dh) / min(mup_dh)
sp_spread = max(sp_dh) / min(sp_dh)
print(f'\nμP |Δh| 跨宽度倍数 {mup_spread:.2f}x (越接近1越好)')
print(f'SP |Δh| 跨宽度倍数 {sp_spread:.2f}x')
assert mup_spread < sp_spread, 'μP 的坐标量级应比 SP 更稳(更接近水平线)'
print('✅ coordinate check：μP 的 Δactivation 跨宽度更平 —— 这就是「曲线水平=μP 正确」的判据')

## 6 · LR transfer：最优 lr 在 μP 下不随宽度漂移

终极验证：在每个宽度上**网格搜最优 base lr**。**SP：最优 lr 随宽度漂移(往下走)；μP：最优 lr 在所有宽度上相同** → 小模型调好直接用到大模型。

In [ ]:
def train_loss(width, base_lr, param, seed, steps=15, batch=32, d_in=8):
    g = np.random.default_rng(seed)
    if param == 'sp':
        W1 = g.standard_normal((width, d_in)) * np.sqrt(1/d_in)
        W2 = g.standard_normal((1, width)) * np.sqrt(1/width)
        lr1 = base_lr; lr2 = base_lr
    else:
        W1 = g.standard_normal((width, d_in)) * np.sqrt(1/d_in)
        W2 = g.standard_normal((1, width)) * (1.0/width)
        lr1 = base_lr * width; lr2 = base_lr / width
    tr = np.random.default_rng(999)
    Wtrue = tr.standard_normal(d_in)
    X = tr.standard_normal((batch, d_in)); T = X @ Wtrue
    with np.errstate(all='ignore'):            # SP 高 lr 会发散，静默(下面用 1e9 兜底)
        for _ in range(steps):
            pre = X @ W1.T; H = np.maximum(pre, 0); Y = (H @ W2.T)[:, 0]
            dY = (Y - T) / batch
            gW2 = dY[None, :] @ H; gH = dY[:, None] * W2
            gpre = gH * (pre > 0); gW1 = gpre.T @ X
            W1 = W1 - lr1 * gW1; W2 = W2 - lr2 * gW2
            if not np.isfinite(W1).all():
                return 1e9
        H = np.maximum(X @ W1.T, 0); Y = (H @ W2.T)[:, 0]
    return float(np.mean((Y - T) ** 2))

lrs = np.logspace(-3, 0.5, 12)
for param in ['sp', 'mup']:
    print(f'=== {param.upper()} : 每个宽度的最优 base lr (平均 8 seed) ===')
    best_lrs = []
    for w in [16, 64, 256, 1024]:
        losses = [np.mean([train_loss(w, lr, param, s) for s in range(8)]) for lr in lrs]
        best = lrs[int(np.argmin(losses))]; best_lrs.append(best)
        print(f'  width {w:5d}: 最优 base lr = {best:.4f}')
    if param == 'mup':
        # μP: 所有宽度最优 lr 相同
        assert len(set(best_lrs)) == 1, 'μP 下最优 base lr 应在所有宽度上相同(可迁移)'
        print('  -> μP: 所有宽度最优 lr 相同 ✓ 小模型调好直接迁到大模型')
    else:
        # SP: 最优 lr 随宽度下降
        assert best_lrs[-1] < best_lrs[0], 'SP 下最优 lr 应随宽度漂移(下降)'
        print('  -> SP: 最优 lr 随宽度往下漂 ✗ 不能直接迁移')
    print()
print('✅ LR transfer 实证：μP 让最优学习率跨宽度不变 —— μTransfer 的全部价值所在')

---
## ✏️ 练习 1：μP 缩放因子

实现 `hidden_lr_scale(base_width, target_width, optimizer)`：返回隐藏层学习率从 base 到 target 的乘法因子。

规则：SGD 下 ∝ width(即 `target/base`)，Adam 下为常数 1.0。

In [ ]:
def hidden_lr_scale(base_width, target_width, optimizer='sgd'):
    # TODO: SGD -> target_width/base_width; Adam -> 1.0
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
assert hidden_lr_scale(128, 512, 'sgd') == 4.0, 'SGD 下 4x 宽 -> lr ×4'
assert hidden_lr_scale(128, 512, 'adam') == 1.0, 'Adam 下常数'
assert hidden_lr_scale(256, 256, 'sgd') == 1.0, '同宽度不缩放'
print('✅ 练习 1 通过：μP 隐藏层 lr 缩放正确')

## ✏️ 练习 2：coordinate check 判据

实现 `is_mup_correct(coord_magnitudes, tol=2.0)`：`coord_magnitudes` 是不同宽度下测得的同一指标值列表。

若 `max/min < tol`(各宽度基本水平)返回 True(μP 可能正确)，否则 False(漂移)。

In [ ]:
def is_mup_correct(coord_magnitudes, tol=2.0):
    # TODO: 返回 max(coord_magnitudes)/min(coord_magnitudes) < tol
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
flat = [1.0, 1.1, 0.95, 1.05]      # 水平 -> μP 正确
drift = [0.1, 0.5, 2.0, 8.0]       # 倾斜 -> 漂移
assert is_mup_correct(flat) is True, '水平曲线应判为正确'
assert is_mup_correct(drift) is False, '倾斜曲线应判为漂移'
# 用第 5 节真实测得的数据
assert is_mup_correct(mup_dh) == (max(mup_dh)/min(mup_dh) < 2.0)
print('✅ 练习 2 通过：会用「曲线是否水平」判 μP 实现')

## ✏️ 练习 3：lr transfer 节省的算力

μP 让你在小模型上搜 lr。实现 `tuning_savings(n_lrs, small_flops, big_flops)`：
- 无 μP：在大模型上搜 n_lrs 次 = `n_lrs * big_flops`；
- 有 μP：小模型搜 n_lrs 次 + 大模型训 1 次 = `n_lrs * small_flops + big_flops`。

返回 (无μP成本, 有μP成本, 节省倍数)。

In [ ]:
def tuning_savings(n_lrs, small_flops, big_flops):
    # TODO: 算两种成本与节省倍数(无μP/有μP)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
# 在 0.04B 代理模型搜 12 个 lr，再训 70B 大模型一次
small = 6 * 0.04e9 * (20*0.04e9)
big = 6 * 70e9 * (20*70e9)
no_mup, with_mup, savings = tuning_savings(12, small, big)
print(f'无μP成本 {no_mup:.2e} FLOPs')
print(f'有μP成本 {with_mup:.2e} FLOPs')
print(f'节省 {savings:.1f}x')
assert no_mup == 12 * big
assert with_mup == 12 * small + big
assert savings > 5, 'μP 应大幅省调参算力'
print('✅ 练习 3 通过：μP 把调参成本从「12个大模型」降到「~1个大模型」')

## ✏️ 练习 4：Δoutput 的宽度标度

用第 2 节的 `delta_output`，实现 `width_exponent(param)`：在两个宽度(256, 1024)测 Δy，估计 `Δy ∝ width^p` 的指数 p。

p ≈ `log(Δy_big/Δy_small) / log(1024/256)`。SP 应得 p≈1(线性增长)，μP 应得 p≈0(持平)。

In [ ]:
def width_exponent(param):
    # TODO: w1,w2=256,1024; d1,d2=delta_output(w1,param),delta_output(w2,param)
    #       return log(d2/d1)/log(w2/w1)
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
p_sp = width_exponent('sp')
p_mup = width_exponent('mup')
print(f'SP : Δy ∝ width^{p_sp:.2f} (应≈1，线性增长)')
print(f'μP : Δy ∝ width^{p_mup:.2f} (应≈0，持平)')
assert p_sp > 0.7, 'SP 的 Δy 应近似随宽度线性增长(指数≈1)'
assert abs(p_mup) < 0.4, 'μP 的 Δy 应近似持平(指数≈0)'
print('✅ 练习 4 通过：量化了 SP(p≈1) 与 μP(p≈0) 的本质区别')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def hidden_lr_scale(base_width, target_width, optimizer='sgd'):
    return (target_width / base_width) if optimizer == 'sgd' else 1.0

In [ ]:
# 练习 2 参考答案
def is_mup_correct(coord_magnitudes, tol=2.0):
    return max(coord_magnitudes) / min(coord_magnitudes) < tol

In [ ]:
# 练习 3 参考答案
def tuning_savings(n_lrs, small_flops, big_flops):
    no_mup = n_lrs * big_flops
    with_mup = n_lrs * small_flops + big_flops
    return no_mup, with_mup, no_mup / with_mup

In [ ]:
# 练习 4 参考答案
def width_exponent(param):
    w1, w2 = 256, 1024
    d1, d2 = delta_output(w1, param), delta_output(w2, param)
    return np.log(d2 / d1) / np.log(w2 / w1)

---
## 🧪 真实数据胶囊：μP × Chinchilla 一起规划一次预训练

把 μP(超参怎么迁)与 Chinchilla(规模怎么分)结合，规划一次真实预训练：给定算力预算，Chinchilla 定 N 和 D，μP 让你在小代理上搜超参再迁过去。算这套组合拳的全貌。

In [ ]:
def plan_pretraining(compute_budget_flops, ratio=20):
    '''用 Chinchilla 由算力定 N,D；返回规划字典。'''
    import math
    N = math.sqrt(compute_budget_flops / (6 * ratio))   # C=6*ratio*N^2
    D = ratio * N
    return {'params_N': N, 'tokens_D': D, 'check': 6 * N * D}

budget = 1e23     # 一次大型预训练的算力量级
plan = plan_pretraining(budget)
print(f'算力预算 {budget:.0e} FLOPs')
print(f'Chinchilla 最优: N≈{plan["params_N"]/1e9:.1f}B 参数, D≈{plan["tokens_D"]/1e9:.0f}B token')
print(f'(校验 6ND = {plan["check"]:.2e} ≈ 预算 ✓)')

# 超参在 N/1000 的代理模型上搜，再用 μP 迁移
proxy_N = plan['params_N'] / 1000
proxy_compute = 6 * proxy_N * (20 * proxy_N)
print(f'\nμP 调参: 在 {proxy_N/1e6:.1f}M 代理模型上搜超参')
print(f'代理训练成本仅为大模型的 {proxy_compute/budget*100:.4f}%')
assert abs(plan['check'] - budget) / budget < 0.01, 'Chinchilla 规划应自洽'
assert proxy_compute < budget * 0.01, 'μP 代理调参应远便宜于大模型'
print('✅ 胶囊：Chinchilla 定规模 + μP 迁超参 = 现代预训练「放大」环节的标准打法')

**🧪 胶囊练习**：实现 `proxy_cost_fraction(big_N, shrink, ratio=20)`：代理模型是大模型的 `1/shrink` 参数，返回代理一次训练成本占大模型的比例。

提示：成本 ∝ N²(因 D∝N)，所以比例 = `(1/shrink)²`。

In [ ]:
def proxy_cost_fraction(big_N, shrink, ratio=20):
    # TODO: 代理 N = big_N/shrink; 成本∝N^2 -> 比例 = (1/shrink)^2
    raise NotImplementedError

In [ ]:
# 自测
frac = proxy_cost_fraction(70e9, 100)
print(f'代理是大模型 1/100 参数 -> 一次训练成本占 {frac*100:.4f}%')
assert abs(frac - 1e-4) < 1e-9, '1/100 参数 -> 成本 1/10000'
print('✅ 胶囊练习通过：代理模型小一个量级，调参成本小两个量级')

In [ ]:
# 📖 胶囊参考答案
def proxy_cost_fraction(big_N, shrink, ratio=20):
    return (1.0 / shrink) ** 2

### 小结
- **病根**：SP + 固定 lr 下，一步更新对输出的冲击 Δy ∝ 宽度 → 最优 lr 随宽度漂移 → 不能把小模型 lr 用到大模型。
- **μP**：重新缩放各层 init 方差与 lr(隐藏 init∝1/d、输出 init∝1/d²、lr 按优化器调、注意力用 1/d) → Δy 在所有宽度=O(1) → 最优 lr 不漂移、**可迁移**。
- **coordinate check**：测各层坐标量级随宽度是否水平，是验证 μP 实现的客观判据(曲线水平=正确)。
- **μTransfer + Chinchilla**：scaling law 定规模、μP 迁超参，在小代理上调好直接放大 —— 把调参成本从「N 个大模型」降到「~1 个」。

下一站：**模块 04 · 训练稳定性** —— 超参定了，怎么让这几个月的训练别在某个深夜突然 loss 爆炸、前功尽弃？